# NB1 — Baseline: variance propagation and the α = 1/√2 control

**Purpose.**
1. Reproduce the textbook claims this project builds on: with He initialisation a residual
   block doubles the activation variance (UDL §11.3), and rescaling by 1/√2 keeps it
   constant (UDL Fig. 11.6b).
2. Derive and check the initialisation σ²(α) = 2(1−α²)/D, and the per-block calibration that
   holds the forward variance fixed for any skip weight α, including a check of the backward pass.
3. Train the control condition used in NB2 (32 blocks, α = 1/√2) with ≥ 3 seeds and analyse
   its variability.

**Contents.** 0 Setup · 1 Model · 2 Reproducing variance doubling · 3 Derivation of σ²(α) ·
4 Finite-width spread and calibration · 5 Training the baseline · 6 Variability ·
7 Decisions carried into NB2

In [ ]:
QUICK = True  # Submission target: cached runs in under five minutes on the lab kernel.

## 0. Setup

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import config as C
from utils import (env_report, get_device, set_seed, make_generator, load_fashion_mnist,
                   iterate_minibatches, evaluate, save_result, result_exists, load_results,
                   summarise, plot_mean_sd, plot_seed_points)

T0 = time.time()
ENV = env_report()
DEVICE = get_device()
DATA = load_fashion_mnist(device=DEVICE, n_val=C.N_VAL, split_seed=C.SPLIT_SEED,
                          n_calib=C.N_CALIB, calib_seed=C.CALIB_SEED)
print({k: tuple(v.shape) for k, v in DATA.items() if torch.is_tensor(v)})

## 1. Model

Pre-activation residual block (UDL Fig. 11.5b): `h ← α·h + W·ReLU(h)`, no bias and no
normalisation, as specified in the design.

_TODO (owner): explain in a few sentences why the activation comes before the linear layer,
and why bias and normalisation are left out._

In [ ]:
class ResMLP(nn.Module):
    """Pre-activation residual MLP.

    input  : Linear(784 -> D, bias=False)                  -> h_0
    block l: h_{l+1} = alpha * h_l + W_l(ReLU(h_l))         W_l = Linear(D -> D, bias=False)
    output : Linear(D -> 10, bias=False) applied to h_L      (logits)

    forward(x, return_hidden=False) -> logits, or (logits, [h_0, ..., h_L]) if return_hidden.
    """

    def __init__(self, depth: int, width: int = C.WIDTH, alpha: float = 0.0):
        super().__init__()
        # TODO: create the input layer, `depth` block layers (nn.ModuleList) and the output layer
        raise NotImplementedError

    def forward(self, x: torch.Tensor, return_hidden: bool = False):
        # TODO
        raise NotImplementedError

In [ ]:
@torch.no_grad()
def init_weights(model: ResMLP, alpha: float, sigma2_block: float | None = None) -> None:
    """Initialise all weights in place with zero-mean Gaussians.

    input layer: N(0, 1/784); every block: N(0, sigma2_block), default 2(1 - alpha^2)/D;
    output layer: N(0, 1/D). Pass sigma2_block = 2/D for plain He initialisation (Section 2).
    """
    # TODO
    raise NotImplementedError

## 2. Reproducing UDL §11.3: variance doubling and the 1/√2 fix

Forward pass only, no training. For ≥ 10 networks (different seeds), push `DATA["x_calib"]`
through the network and record E[h_l²] / E[h_0²] for every layer l:

- (a) α = 1 with He initialisation (σ² = 2/D): expected to grow by ≈ ×2 per block;
- (b) α = 1/√2 with σ² = 1/D: expected to stay ≈ 1.

Plot mean ± sd over networks against l (log scale for (a)). Report the per-block factor as an
average over networks: a single network deviates because of finite width (Section 4).

In [ ]:
# TODO: variance-propagation experiment for (a) and (b); plot with plot_mean_sd

## 3. Deriving σ²(α)

_TODO (owner): derivation in your own words, forward and backward._

- Forward: second moment of the branch, why the cross term vanishes, the per-block factor.
- Backward: the same factor for the gradient with respect to h, using that W is square.
- Special cases: α = 0 (He), α = 1/√2 (UDL Fig. 11.6b), α = 1 with He (doubling).

Target result: $\sigma^2(\alpha) = \dfrac{2(1-\alpha^2)}{D}$.

## 4. Finite-width spread and calibration

The derivation holds in expectation over the weights. Show how much single networks deviate
(final-layer variance across ≥ 10 networks, for each α in `C.ALPHAS[32]`), then calibrate each
network so its forward variance is exactly preserved on the calibration batch.

Checks to report for every condition (both depths):
1. forward: per-layer E[h_l²] ratio after calibration = 1.000 (and close to 1 on held-out data);
2. backward: per-layer second moment of ∂loss/∂h_l compared across conditions
   (hint: `h.retain_grad()` on the hidden states returned by `forward(..., return_hidden=True)`).

In [ ]:
@torch.no_grad()
def calibrate(model: ResMLP, x_calib: torch.Tensor, alpha: float) -> list[float]:
    """Rescale each block's W so that E[h_{l+1}^2] = E[h_l^2] exactly on x_calib.

    For block l with f = W_l(ReLU(h_l)):  a = E[f^2], b = E[h_l * f], c = E[h_l^2]
        s = (-alpha * b + sqrt(alpha^2 * b^2 + (1 - alpha^2) * a * c)) / a,   W_l <- s * W_l
    Process the blocks in order, so block l+1 sees the already-calibrated h_{l+1}.
    Returns the list of scale factors s (they should all be close to 1).
    """
    # TODO
    raise NotImplementedError

In [ ]:
# TODO: spread before calibration, forward check after calibration, backward check

## 5. Training the baseline (32 blocks, α = 1/√2)

Protocol: sweep `C.LR_GRID` with `C.TUNE_SEED`, select the learning rate by final training loss
(rule in `hypotheses.md`), then rerun with `C.FINAL_SEEDS` and the test set.

QUICK mode target: load cached runs and train one small live example.

In [ ]:
def train_one_run(depth: int, alpha: float, lr: float, seed: int, epochs: int = C.EPOCHS,
                  width: int = C.WIDTH, data: dict = None, save: bool = True,
                  with_test: bool = False, n_train: int | None = None, tag: str = "") -> dict:
    """Train one network; return (and by default save) its result record.

    set_seed(seed) -> ResMLP(depth, width, alpha) -> init_weights -> calibrate on data["x_calib"]
    -> torch.optim.SGD(lr=lr, momentum=C.MOMENTUM), batches of C.BATCH_SIZE shuffled with
       make_generator(seed)
    -> after every epoch: evaluate() on the full training and validation sets
    -> if the loss becomes NaN/inf, stop and set diverged = True (keep the record).

    Record keys: depth, width, alpha, lr, seed, epochs, train_loss, val_loss, train_acc, val_acc
    (lists per epoch; train_loss[C.EARLY_EPOCH - 1] is the early-training metric),
    final_train_loss, diverged, seconds, and test_loss / test_acc when
    with_test (final runs only). n_train trains on a subset (QUICK live example; then save=False).
    """
    data = DATA if data is None else data
    # TODO
    raise NotImplementedError

In [ ]:
if QUICK:
    baseline = load_results(depth=C.DEPTH_MAIN, alpha=C.ALPHA_BASELINE)
    print(f"loaded {len(baseline)} cached baseline runs")
    # TODO: one small live example, e.g. train_one_run(..., epochs=1, n_train=10_000, save=False)
else:
    pass  # TODO: lr sweep (skip runs where result_exists), select lr, final seeds with with_test=True

## 6. Baseline results and variability

_TODO:_ training-loss curves (mean ± sd over the final seeds), final loss mean ± sd, and a short
comment on the spread. State which parts of the experiment the random state affects (weight
initialisation and data order) and which are fixed (data split and calibration batch).

In [ ]:
# TODO: plots and summary table (summarise(...) from utils)

## 7. Decisions carried into NB2

_TODO:_
1. Has the baseline loss flattened by epoch `C.EPOCHS`?
2. What is the threshold τ for the usable-learning-rate range, and how was it chosen?
3. Range check: 32 blocks, α ∈ {0.5, 0.9}, two or three learning rates. Are the two
   distinguishable at epoch `C.EARLY_EPOCH` or `C.EPOCHS`? If not, discuss whether 32 blocks
   is deep enough before freezing the design. (This checks the measuring range only; it is
   not evidence for any hypothesis.)

Record the decisions in `hypotheses.md` before the main sweep.

In [ ]:
print(f"Total runtime: {time.time() - T0:.0f} s  (QUICK = {QUICK}; must stay under 300 s when QUICK)")